# Advanced Agent Workflow: Task Execution with Sequential Thinking and Computer Use

This notebook demonstrates an advanced workflow where a cohort of agents is generated from customer profiles, and then each agent is tasked with solving a problem using a structured thinking process. Each agent will:

1.  Be generated from a predefined customer profile.
2.  Receive a task to perform on a desktop, which it can interact with using natural language.
3.  Use the `sequential_thinking` tool to break down the task and plan its actions.
4.  Use the `computer_use` tool to interact with the desktop in front of it.
5.  Reflect on its progress and decide whether to continue or stop.
6.  Log its entire thought process and actions to a file.

Finally, we will analyze the results to determine the success rate and gather insights from the agents' final reflections.

## IMPORTANT: Restart the Kernel

After installing or updating libraries, you **must** restart the Colab kernel for the changes to take effect. Go to **Runtime > Restart session** in the menu above.

## Setup: API Key

This example uses the `browser-use` library, which requires an API key for its cloud service. Please follow these steps to set up your key:

1.  Go to [cloud.browser-use.com](https://cloud.browser-use.com/settings?tab=api-keys) to get your API key.
2.  In your Colab notebook, go to the **Secrets** tab (the key icon on the left) and add a new secret named `BROWSER_USE_API_KEY` with your key as the value.

In [ ]:
import os
import sys

# Load the API key from Colab secrets
try:
    from google.colab import userdata
    os.environ['BROWSER_USE_API_KEY'] = userdata.get('BROWSER_USE_API_KEY')
except ImportError:
    print("Not in a Colab environment, skipping secret loading. Make sure BROWSER_USE_API_KEY is set.")

# Ensure the .env file is loaded for local development
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("dotenv not installed, skipping.")

In [ ]:
# Add the project root to the Python path to allow importing tinytroupe
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from tinytroupe.agent.tiny_person import TinyPerson

print("--- Step 1: Persona Generation ---")

# 1. Define Customer Profiles
customer_profiles = [
    {"name": "Alice", "occupation": "Software Developer", "age": 28, "interests": ["AI", "startups", "python"]},
    {"name": "Bob", "occupation": "Marketing Manager", "age": 35, "interests": ["SEO", "content creation", "analytics"]},
    {"name": "Charlie", "occupation": "Data Scientist", "age": 31, "interests": ["machine learning", "statistics", "big data"]},
    {"name": "Diana", "occupation": "UX Designer", "age": 29, "interests": ["user research", "prototyping", "visual design"]},
    {"name": "Eve", "occupation": "Product Manager", "age": 40, "interests": ["agile methodologies", "roadmapping", "user feedback"]},
    {"name": "Frank", "occupation": "Student", "age": 21, "interests": ["deep learning", "computer vision", "robotics"]},
    {"name": "Grace", "occupation": "Technical Writer", "age": 33, "interests": ["documentation", "API reference", "knowledge bases"]},
    {"name": "Heidi", "occupation": "DevOps Engineer", "age": 38, "interests": ["CI/CD", "cloud infrastructure", "automation"]},
    {"name": "Ivan", "occupation": "Security Analyst", "age": 45, "interests": ["threat modeling", "penetration testing", "cryptography"]},
    {"name": "Judy", "occupation": "CEO", "age": 52, "interests": ["business strategy", "leadership", "innovation"]},
]

# 2. Mock Social Media Data Retrieval
def get_mock_social_data(profile):
    return {
        "linkedin_summary": f"{profile['name']} is a passionate {profile['occupation']}.",
        "x_bio": f"Tweeting about {', '.join(profile['interests'])}"
    }

# 3. Generate Agents
def generate_agents(profiles):
    agents = []
    for profile in profiles:
        if TinyPerson.has_agent(profile['name']):
            TinyPerson.all_agents.pop(profile['name'])
            
        social_data = get_mock_social_data(profile)
        agent = TinyPerson(name=profile['name'])
        
        agent.define("age", profile['age'])
        agent.define("occupation", profile['occupation'])
        agent.define("interests", profile['interests'])
        agent.define("persona_summary", social_data)
        
        agents.append(agent)
        print(f"Generated agent: {agent.name}")
        
    return agents

print("--- Generating 10 agents based on customer profiles ---")
agent_cohort = generate_agents(customer_profiles)
print(f"\nSuccessfully generated {len(agent_cohort)} agents.")

## Step 2: Main Simulation Loop

In [ ]:
import logging
from tinytroupe.utils.logger import setup_agent_logger

LOGS_DIR = "logs"
os.makedirs(LOGS_DIR, exist_ok=True)

def run_simulation(agents):
    WEBSITE_URL = "https://www.wikipedia.org/"
    task_description = (f"Your primary goal is to explore the website at {WEBSITE_URL}. "
                        "Start by navigating to the homepage. Your task is to find the article about 'Artificial Intelligence' and summarize the first paragraph.")

    for agent in agents:
        print(f"\n--- Starting task for agent: {agent.name} ---")
        
        log_file = os.path.join(LOGS_DIR, f"{agent.name}.log")
        setup_agent_logger(agent.name, log_file)
        
        system_prompt = f"""
        You are {agent.name}, a {agent.get('occupation')}. You have been given the following task:
        {task_description}

        To accomplish this, you must use a structured thinking process. For each step, you must:
        1.  **Plan:** Use the `sequential_thinking` tool to decide your next action.
        2.  **Act:** Execute the action by calling the `computer_use` tool with a natural language command (e.g., 'Go to wikipedia.org', 'Type 'Artificial Intelligence' into the search bar').
        3.  **Reflect:** After each action, use the `THINK` action to reflect on the outcome.

        When you have found the answer, your final action must be `DONE`.
        """
        agent.define("system_prompt_override", system_prompt)

        agent.think(f"I have received the following task: {task_description}")
        agent.act(until_done=True, communication_display=False)
        
        print(f"--- Task finished for agent: {agent.name}. See {log_file} for details. ---")

run_simulation(agent_cohort)

## Step 3: Results Analysis and Reporting

In [ ]:
def analyze_results(agents):
    print("\n--- Step 3: Results Analysis and Reporting ---")
    
    successful_agents = 0
    total_agents = len(agents)
    
    for agent in agents:
        print(f"\n--- Analysis for Agent: {agent.name} ---")
        
        last_memories = agent.episodic_memory.retrieve(last_n=5, include_omission_info=False)
        final_thought = "No final thought found."
        task_accomplished = False
        
        print("Final Interactions:")
        for memory in last_memories:
            if memory.get('type') == 'action':
                action = memory.get('content', {}).get('action', {})
                action_type = action.get('type')
                action_content = action.get('content', '')
                print(f"  - Action: {action_type}")
                if action_content:
                    print(f"    Content: {action_content}")
                
                if action_type == 'THINK' and ("summarized" in action_content.lower() or "summary" in action_content.lower()):
                    task_accomplished = True
                    final_thought = action_content
        
        if task_accomplished:
            successful_agents += 1
            print(f"\nOutcome: Success")
        else:
            print(f"\nOutcome: Did not complete or gave up")
        
        print(f"Final Reflection: {final_thought}")

    accomplishment_rate = (successful_agents / total_agents) * 100 if total_agents > 0 else 0
    print(f"\n--- Overall Summary ---")
    print(f"Total Agents: {total_agents}")
    print(f"Successful Agents: {successful_agents}")
    print(f"Accomplishment Rate: {accomplishment_rate:.2f}%")

analyze_results(agent_cohort)